# Continuous Affect Manifold Hypothesis

**Claim under test.** The LLM does not natively represent affect as eight discrete primitives (`joy`, `trust`, `fear`, ...).  Instead, affect lives on a **low-rank, continuous, linear manifold with a shared intensity axis**, and the eight Plutchik labels are a coarse, human-imposed partition of it.

**Evidence we will look for:**
1. Eight emotions ARE linearly decodable (already shown: bal-acc 0.817 at layer 13 in `analysis/emotion_intensity/summary.txt`). The interesting question is *what underlies that decodability*.
2. The 8 emotion centroids occupy a **low-rank continuous subspace**, not 8 independent directions.
3. **Pairwise emotion direction vectors are relative**, not absolute — `v(joy→fear)` is approximately the same vector regardless of source text or intensity.
4. **Intensity is a shared continuous axis** — `low < medium < high` aligned along a single direction shared across emotions.

If all four hold, the discrete-label view is a useful read-out but not the actual representational primitive.

**Input.** `activation/emotion_rewrites/emotion_intensity_residual_stream.npy` with shape `(N, 8, 3, 6, D)`; primary layer **13**, robustness checks at 10 and 16.

**Outputs.** `analysis/emotion_intensity/continuous_affect/` (CSVs + `summary.json`).

## 0. Imports & configuration

In [1]:
from __future__ import annotations

import gc
import json
import warnings
from pathlib import Path
from itertools import combinations, permutations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeClassifier, Ridge
from sklearn.metrics import (
    adjusted_rand_score, normalized_mutual_info_score,
    balanced_accuracy_score, silhouette_score,
)
from sklearn.model_selection import GroupKFold
from scipy.spatial.distance import cosine as cosine_dist
from scipy.optimize import least_squares

try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

REPO_ROOT = Path("../..").resolve()
ACT_DIR   = REPO_ROOT / "activation" / "emotion_rewrites"
NPY_PATH  = ACT_DIR / "emotion_intensity_residual_stream.npy"
INFO_PATH = ACT_DIR / "emotion_intensity_residual_stream_info.json"
JSONL_PATH = REPO_ROOT / "dataset" / "emotion_rewrites" / "emotion_rewrites.jsonl"
OUT_DIR   = REPO_ROOT / "analysis" / "emotion_intensity" / "continuous_affect"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
PRIMARY_LAYER = 13          # best layer per existing summary.txt
ROBUST_LAYERS = [10, 16]    # headline-only robustness checks

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150})
EMO_COLORS = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45",
]
INT_COLORS = {"low": "#4575b4", "medium": "#fdae61", "high": "#d73027"}

print(f"OUT_DIR: {OUT_DIR}")
print(f"UMAP available: {HAS_UMAP}")

/home/maplesugano/proj/EmotionEngine_v2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

## Phase A — Three views of the activations

We keep three views and pick the appropriate one per analysis.

| view | formula | meaning |
|------|---------|---------|
| `X_raw` | $X_{s,e,i}$ | absolute affect space (used for intensity / VAD) |
| `R_source` | $X_{s,e,i} - \frac{1}{8\cdot 3}\sum_{e',i'} X_{s,e',i'}$ | base-text removed, **emotion & intensity preserved** — primary view for the manifold |
| `R_source_intensity` | $X_{s,e,i} - \frac{1}{8}\sum_{e'} X_{s,e',i}$ | strict relative-emotion contrast (sensitivity check for Phase C) |

In [ ]:
with open(INFO_PATH) as f:
    INFO = json.load(f)

LAYER_INDICES:   list[int] = INFO["layer_indices"]
EMOTION_ORDER:   list[str] = INFO["emotion_order"]
INTENSITY_ORDER: list[str] = INFO["intensity_order"]
SOURCE_IDS:      list[str] = INFO["source_ids"]
SHAPE                       = tuple(INFO["shape"])
DTYPE                       = INFO.get("dtype", "float32")
N, N_EMO, N_INT, N_LAYERS, D = SHAPE

assert N == len(SOURCE_IDS), "source_ids length mismatch"
assert N_EMO == 8 and N_INT == 3 and N_LAYERS == 6

LAYER_TO_IDX = {l: i for i, l in enumerate(LAYER_INDICES)}
EMO_TO_IDX   = {e: i for i, e in enumerate(EMOTION_ORDER)}

act = np.memmap(str(NPY_PATH), dtype=DTYPE, mode="r", shape=SHAPE)
print(f"Activation tensor : {act.shape}  dtype={DTYPE}")
print(f"Layer indices     : {LAYER_INDICES}")
print(f"Emotion order     : {EMOTION_ORDER}")
print(f"Intensity order   : {INTENSITY_ORDER}")
print(f"# sources (N)     : {N}")
print(f"Hidden dim (D)    : {D}")

In [ ]:
def build_views(layer: int) -> dict[str, np.ndarray]:
    """Return the three views for a given hook layer as float32 arrays of shape (N, 8, 3, D)."""
    li = LAYER_TO_IDX[layer]
    # Read the full layer slice into RAM once: (N, 8, 3, D) float32 ~= N*96*D*4 bytes.
    # For N=22782, D=4096 -> ~34 GB; we cannot load full. So compute in source chunks.
    X_raw = np.empty((N, N_EMO, N_INT, D), dtype=np.float32)
    CHUNK = 256
    for s0 in range(0, N, CHUNK):
        s1 = min(N, s0 + CHUNK)
        X_raw[s0:s1] = np.asarray(act[s0:s1, :, :, li, :], dtype=np.float32)
    # Per-source mean over (e, i): shape (N, 1, 1, D)
    mu_source = X_raw.mean(axis=(1, 2), keepdims=True)
    R_source = X_raw - mu_source
    # Per-(source, intensity) mean over e: shape (N, 1, 3, D)
    mu_source_intensity = X_raw.mean(axis=1, keepdims=True)
    R_source_intensity = X_raw - mu_source_intensity
    # Sanity asserts
    assert np.max(np.abs(R_source.mean(axis=(1, 2)))) < 1e-3, "R_source per-source mean not ~0"
    assert np.max(np.abs(R_source_intensity.mean(axis=1))) < 1e-3, "R_source_intensity per-(s,i) mean not ~0"
    return {
        "X_raw": X_raw,
        "R_source": R_source,
        "R_source_intensity": R_source_intensity,
    }

print("Estimating per-layer RAM:", f"{N * N_EMO * N_INT * D * 4 / 1e9:.1f} GB per view (3 views ≈ {3 * N * N_EMO * N_INT * D * 4 / 1e9:.1f} GB)")

In [ ]:
# Memory consideration: full 22782 × 8 × 3 × 4096 × float32 per view ≈ 8.95 GB; 3 views ≈ 27 GB.
# We subsample sources to keep this tractable. Sampling is uniform over source_ids; analyses are
# averages over (s, i) so a few-thousand-source subsample gives well-converged means.
MAX_SOURCES = 2500  # ~1.0 GB per view; 3 views ~3 GB. Increase if RAM permits.
rng = np.random.default_rng(SEED)
if N > MAX_SOURCES:
    SRC_SUBSAMPLE = np.sort(rng.choice(N, size=MAX_SOURCES, replace=False))
else:
    SRC_SUBSAMPLE = np.arange(N)
N_SUB = len(SRC_SUBSAMPLE)
SRC_ID_SUB = [SOURCE_IDS[i] for i in SRC_SUBSAMPLE]
print(f"Using {N_SUB} of {N} sources (RAM control). Seed={SEED}.")

def build_views_sub(layer: int) -> dict[str, np.ndarray]:
    li = LAYER_TO_IDX[layer]
    X_raw = np.empty((N_SUB, N_EMO, N_INT, D), dtype=np.float32)
    CHUNK = 256
    for s0 in range(0, N_SUB, CHUNK):
        s1 = min(N_SUB, s0 + CHUNK)
        idxs = SRC_SUBSAMPLE[s0:s1]
        X_raw[s0:s1] = np.asarray(act[idxs, :, :, li, :], dtype=np.float32)
    mu_s   = X_raw.mean(axis=(1, 2), keepdims=True)
    R_s    = X_raw - mu_s
    mu_si  = X_raw.mean(axis=1, keepdims=True)
    R_si   = X_raw - mu_si
    assert np.max(np.abs(R_s.mean(axis=(1, 2)))) < 1e-3
    assert np.max(np.abs(R_si.mean(axis=1))) < 1e-3
    return {"X_raw": X_raw, "R_source": R_s, "R_source_intensity": R_si}

VIEWS = build_views_sub(PRIMARY_LAYER)
for k, v in VIEWS.items():
    print(f"  {k:22s}  shape={v.shape}  dtype={v.dtype}  ~{v.nbytes/1e9:.2f} GB")

## Phase B — Centroid low-rank continuous subspace

Goal: the 8 emotion centroids should lie on a 2-3 dimensional subspace (not be in 8 independent directions). We use the **primary view `R_source`** (base text removed, emotion + intensity preserved).

In [ ]:
def centroid_geometry(R: np.ndarray) -> dict:
    """R shape (N, 8, 3, D); compute 8 centroids over (s, i), SVD, participation ratio."""
    C = R.mean(axis=(0, 2))                                  # (8, D)
    # Centre the centroid matrix (the grand mean over emotions should be ~0 already given R_source)
    Cc = C - C.mean(axis=0, keepdims=True)
    U, S, Vt = np.linalg.svd(Cc, full_matrices=False)        # S length min(8, D) = 8
    var = S ** 2
    evr = var / var.sum()
    # Participation ratio of the 8 centroids in singular-value space
    pr  = (var.sum() ** 2) / (var ** 2).sum()
    return {"C": C, "Cc": Cc, "S": S, "evr": evr, "pr": pr, "U": U, "Vt": Vt}

geom = centroid_geometry(VIEWS["R_source"])
print(f"Singular values (8): {np.round(geom['S'], 3)}")
print(f"Explained variance : {np.round(geom['evr'], 3)}")
print(f"  cumulative       : {np.round(np.cumsum(geom['evr']), 3)}")
print(f"Participation ratio of centroids : {geom['pr']:.3f}  (max possible = 8.0)")
print(f"  top-2 EVR        : {geom['evr'][:2].sum():.3f}")
print(f"  top-3 EVR        : {geom['evr'][:3].sum():.3f}")

In [ ]:
# Shuffled-emotion null: per (source, intensity), permute emotion labels, then recompute centroids.
def shuffled_centroid_null(R: np.ndarray, n_perm: int = 50, seed: int = SEED) -> dict:
    rng = np.random.default_rng(seed)
    prs, top2 = [], []
    for _ in range(n_perm):
        Rp = R.copy()
        for s in range(Rp.shape[0]):
            for i in range(Rp.shape[2]):
                perm = rng.permutation(8)
                Rp[s, :, i, :] = Rp[s, perm, i, :]
        g = centroid_geometry(Rp)
        prs.append(g["pr"])
        top2.append(g["evr"][:2].sum())
        del Rp
    return {"pr_null": np.array(prs), "top2_null": np.array(top2)}

null = shuffled_centroid_null(VIEWS["R_source"], n_perm=20)
print(f"Null participation ratio  : mean={null['pr_null'].mean():.3f}  std={null['pr_null'].std():.3f}")
print(f"Real participation ratio  : {geom['pr']:.3f}")
print(f"Null top-2 EVR            : mean={null['top2_null'].mean():.3f}  std={null['top2_null'].std():.3f}")
print(f"Real top-2 EVR            : {geom['evr'][:2].sum():.3f}")
# p-value: probability that null PR <= real PR (lower PR = more concentrated subspace)
pr_p   = float((null["pr_null"] <= geom["pr"]).mean())
top2_p = float((null["top2_null"] >= geom["evr"][:2].sum()).mean())
print(f"p(null PR ≤ real)         : {pr_p:.3f}")
print(f"p(null top-2 ≥ real)      : {top2_p:.3f}")

In [ ]:
# Scree plot + 2-D centroid projection on top-2 SVD axes.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
ax.bar(range(1, 9), geom["evr"], color="#4363d8", alpha=0.85)
ax.axhline(0.05, color="gray", ls="--", lw=0.8)
ax.set_xlabel("Singular component"); ax.set_ylabel("Explained variance")
ax.set_title(f"Centroid SVD spectrum (L{PRIMARY_LAYER})  PR={geom['pr']:.2f}")
ax.grid(True, alpha=0.3)

ax = axes[1]
coords2 = geom["Cc"] @ geom["Vt"][:2].T                       # (8, 2)
for ei, emo in enumerate(EMOTION_ORDER):
    ax.scatter(coords2[ei, 0], coords2[ei, 1], color=EMO_COLORS[ei], s=140, edgecolor="k")
    ax.annotate(emo, coords2[ei], fontsize=10, xytext=(6, 4), textcoords="offset points")
ax.axhline(0, color="gray", lw=0.6); ax.axvline(0, color="gray", lw=0.6)
ax.set_xlabel("SV1"); ax.set_ylabel("SV2")
ax.set_title(f"8 emotion centroids on top-2 SVD axes\nTop-2 EVR={geom['evr'][:2].sum():.2f}")
ax.set_aspect("equal", adjustable="datalim"); ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_DIR / "phaseB_centroid_geometry.png", bbox_inches="tight")
plt.show()

In [ ]:
# Plutchik wheel adjacency test: in Plutchik's wheel the canonical order is
#   joy → trust → fear → surprise → sadness → disgust → anger → anticipation → joy.
# This matches EMOTION_ORDER. We project to top-2, fit a circle, compute angles,
# and compare circular order to the canonical permutation against random nulls.
PLUTCHIK_ORDER = ["joy", "trust", "fear", "surprise", "sadness", "disgust", "anger", "anticipation"]
assert PLUTCHIK_ORDER == EMOTION_ORDER, "emotion order in info.json does not match Plutchik canonical"

def fit_circle(xy: np.ndarray) -> tuple[np.ndarray, float, float]:
    """Algebraic Kasa fit: returns (centre, radius, residual RMS)."""
    x, y = xy[:, 0], xy[:, 1]
    A = np.column_stack([2 * x, 2 * y, np.ones_like(x)])
    b = x ** 2 + y ** 2
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy = sol[0], sol[1]
    r = float(np.sqrt(sol[2] + cx ** 2 + cy ** 2))
    rms = float(np.sqrt(np.mean((np.sqrt((x - cx) ** 2 + (y - cy) ** 2) - r) ** 2)))
    return np.array([cx, cy]), r, rms

def circular_kendall_tau(angles: np.ndarray) -> float:
    """Concordance with the canonical order 0..7 of the given angle sequence.
    Returns the maximum Kendall-tau over all 8 cyclic shifts and 2 directions."""
    n = len(angles)
    # sort indices by angle
    order_by_angle = np.argsort(angles)
    target = np.arange(n)
    best = -1.0
    for direction in (1, -1):
        seq = order_by_angle[::direction]
        for shift in range(n):
            rolled = np.roll(seq, shift)
            # Spearman-like correlation between rolled and target
            rank_target = np.argsort(np.argsort(rolled))
            tau = np.corrcoef(rank_target, target)[0, 1]
            best = max(best, tau)
    return float(best)

centre, radius, rms = fit_circle(coords2)
rel_rms = rms / radius if radius > 0 else float("inf")
angles = np.arctan2(coords2[:, 1] - centre[1], coords2[:, 0] - centre[0])
tau_real = circular_kendall_tau(angles)

# Permutation null: shuffle the 8 centroid labels and re-evaluate concordance.
n_perm = 5000
rng_p = np.random.default_rng(SEED)
tau_null = np.empty(n_perm)
for k in range(n_perm):
    perm = rng_p.permutation(8)
    tau_null[k] = circular_kendall_tau(angles[perm])
wheel_p = float((tau_null >= tau_real).mean())

print(f"Circle fit       : centre={centre.round(3)}  radius={radius:.3f}")
print(f"Fit RMS / radius : {rel_rms:.3f}  (smaller = more circular)")
print(f"Kendall-tau vs Plutchik order : {tau_real:.3f}")
print(f"Permutation p-value           : {wheel_p:.4f}  (n_perm={n_perm})")

# Persist Phase B summary numbers.
PHASE_B = {
    "layer": PRIMARY_LAYER,
    "singular_values": geom["S"].tolist(),
    "evr": geom["evr"].tolist(),
    "top2_evr": float(geom["evr"][:2].sum()),
    "top3_evr": float(geom["evr"][:3].sum()),
    "participation_ratio": float(geom["pr"]),
    "pr_null_mean": float(null["pr_null"].mean()),
    "pr_null_std":  float(null["pr_null"].std()),
    "p_pr_le_real": pr_p,
    "top2_null_mean": float(null["top2_null"].mean()),
    "p_top2_ge_real": top2_p,
    "wheel_fit_rel_rms": rel_rms,
    "wheel_kendall_tau": tau_real,
    "wheel_p_value": wheel_p,
}
pd.DataFrame([PHASE_B]).to_csv(OUT_DIR / "phaseB_summary.csv", index=False)
PHASE_B

## Phase C — Pairwise direction relativity

If `v(a→b)` is a **relative** affect vector rather than a difference between absolute primitives, then `X_raw[s,b,i] − X_raw[s,a,i]` should be approximately the same vector for every `(s, i)`, and the triangle equality `v(a→b) + v(b→c) ≈ v(a→c)` should hold.

In [ ]:
def pairwise_direction_stats(X: np.ndarray) -> tuple[pd.DataFrame, dict]:
    """X shape (N, 8, 3, D).
    For each ordered pair (a, b), compute v(s,i) = X[s,b,i] - X[s,a,i],
    its mean direction, and the cosine of every instance against the mean."""
    Nn = X.shape[0]
    rows = []
    # Pre-compute means and per-instance cosines.
    cos_grand = []
    cos_per_intensity_med = {}
    for a, b in permutations(range(8), 2):
        v = X[:, b, :, :] - X[:, a, :, :]                         # (N, 3, D)
        vbar = v.reshape(-1, X.shape[-1]).mean(axis=0)            # (D,)
        vbar_norm = vbar / (np.linalg.norm(vbar) + 1e-12)
        v_flat = v.reshape(-1, X.shape[-1])
        v_norms = np.linalg.norm(v_flat, axis=1) + 1e-12
        cos = (v_flat @ vbar_norm) / v_norms                      # (N*3,)
        rows.append({
            "a": EMOTION_ORDER[a], "b": EMOTION_ORDER[b],
            "mean_cos":   float(cos.mean()),
            "median_cos": float(np.median(cos)),
            "iqr_cos":    float(np.percentile(cos, 75) - np.percentile(cos, 25)),
            "vbar_norm":  float(np.linalg.norm(vbar)),
        })
        cos_grand.append(cos)
    df = pd.DataFrame(rows)
    summary = {
        "mean_median_cos_pairs": float(df["median_cos"].mean()),
        "min_median_cos": float(df["median_cos"].min()),
        "max_median_cos": float(df["median_cos"].max()),
    }
    return df, summary

pair_df, pair_summary = pairwise_direction_stats(VIEWS["X_raw"])
print("Pairwise direction invariance (X_raw, L13):")
for k, v in pair_summary.items():
    print(f"  {k:30s} {v:.3f}")
display_cols = ["a", "b", "median_cos", "iqr_cos", "vbar_norm"]
pair_df.sort_values("median_cos", ascending=False).head(10)[display_cols]

In [ ]:
# Sensitivity: same analysis on R_source_intensity (centring kills the per-(s,i) emotion mean,
# but pairwise differences X[b]-X[a] are invariant to that mean shift -> cosines should match).
pair_df_si, pair_summary_si = pairwise_direction_stats(VIEWS["R_source_intensity"])
print("Sensitivity check on R_source_intensity:")
for k, v in pair_summary_si.items():
    print(f"  {k:30s} {v:.3f}")
diff = (pair_df.set_index(["a", "b"])["median_cos"] - pair_df_si.set_index(["a", "b"])["median_cos"]).abs()
print(f"\nMax |median_cos diff| between views : {diff.max():.4e}   (should be ~0)")

In [ ]:
# Cross-intensity transfer: compute vbar from intensity=medium only, then measure cosine
# of instances at low and high. High cosine = direction is intensity-invariant.
MED_IDX = INTENSITY_ORDER.index("medium")
LOW_IDX = INTENSITY_ORDER.index("low")
HI_IDX  = INTENSITY_ORDER.index("high")

def cross_intensity_transfer(X: np.ndarray) -> pd.DataFrame:
    rows = []
    for a, b in permutations(range(8), 2):
        v_med = X[:, b, MED_IDX, :] - X[:, a, MED_IDX, :]
        vbar = v_med.mean(axis=0)
        vbar_n = vbar / (np.linalg.norm(vbar) + 1e-12)
        for label, ii in [("low", LOW_IDX), ("high", HI_IDX), ("medium", MED_IDX)]:
            v = X[:, b, ii, :] - X[:, a, ii, :]
            cos = (v @ vbar_n) / (np.linalg.norm(v, axis=1) + 1e-12)
            rows.append({
                "a": EMOTION_ORDER[a], "b": EMOTION_ORDER[b],
                "target_intensity": label,
                "median_cos": float(np.median(cos)),
            })
    return pd.DataFrame(rows)

transfer_df = cross_intensity_transfer(VIEWS["X_raw"])
summary_transfer = transfer_df.groupby("target_intensity")["median_cos"].agg(["mean", "min", "max"]).round(3)
print("Cross-intensity transfer (vbar from medium):")
print(summary_transfer)

In [ ]:
# Triangle test: for each ordered triple (a,b,c), measure how close
#   v(a→b) + v(b→c)   is to   v(a→c)
# Use the per-pair mean vectors (averaged over (s, i)) as estimates of each leg.
def triangle_test(X: np.ndarray) -> tuple[pd.DataFrame, dict]:
    # mean direction per ordered pair, (8, 8, D)
    mean_v = np.zeros((8, 8, X.shape[-1]), dtype=np.float32)
    for a in range(8):
        for b in range(8):
            if a == b:
                continue
            mean_v[a, b] = (X[:, b, :, :] - X[:, a, :, :]).reshape(-1, X.shape[-1]).mean(axis=0)
    rows = []
    for a, b, c in permutations(range(8), 3):
        lhs = mean_v[a, b] + mean_v[b, c]
        rhs = mean_v[a, c]
        res = np.linalg.norm(lhs - rhs)
        denom = np.linalg.norm(rhs) + 1e-12
        cos = float(lhs @ rhs / ((np.linalg.norm(lhs) + 1e-12) * denom))
        rows.append({
            "a": EMOTION_ORDER[a], "b": EMOTION_ORDER[b], "c": EMOTION_ORDER[c],
            "res_ratio": float(res / denom),
            "cos":       cos,
        })
    tri = pd.DataFrame(rows)
    summ = {
        "mean_res_ratio":  float(tri["res_ratio"].mean()),
        "p95_res_ratio":   float(np.percentile(tri["res_ratio"], 95)),
        "mean_triangle_cos": float(tri["cos"].mean()),
    }
    return tri, summ

tri_df, tri_summary = triangle_test(VIEWS["X_raw"])
for k, v in tri_summary.items():
    print(f"  {k:25s} {v:.3f}")
print(f"  # triples evaluated     : {len(tri_df)}")
# Heatmap of mean residual ratio aggregated over the middle vertex b.
hm = tri_df.pivot_table(index="a", columns="c", values="res_ratio", aggfunc="mean")
hm = hm.reindex(index=EMOTION_ORDER, columns=EMOTION_ORDER)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(hm, annot=True, fmt=".2f", cmap="viridis_r", ax=ax, cbar_kws={"label": "mean res / |v(a→c)|"})
ax.set_title("Triangle residual ratio  v(a→b)+v(b→c) vs v(a→c)\n(averaged over middle vertex b)")
plt.tight_layout(); fig.savefig(OUT_DIR / "phaseC_triangle_heatmap.png", bbox_inches="tight")
plt.show()

pair_df.to_csv(OUT_DIR / "phaseC_pairwise_directions.csv", index=False)
transfer_df.to_csv(OUT_DIR / "phaseC_cross_intensity_transfer.csv", index=False)
tri_df.to_csv(OUT_DIR / "phaseC_triangle.csv", index=False)
# Free the sensitivity-check view; not used downstream.
del VIEWS["R_source_intensity"]; gc.collect()

PHASE_C = {
    "layer": PRIMARY_LAYER,
    **{f"pair_{k}": v for k, v in pair_summary.items()},
    **{f"transfer_{lab}_mean": float(summary_transfer.loc[lab, "mean"]) for lab in ["low", "medium", "high"]},
    **{f"tri_{k}": v for k, v in tri_summary.items()},
}
PHASE_C

## Phase D — Shared continuous intensity axis

Goal: confirm that there is a single direction along which `low < medium < high` and that this direction is largely shared across the eight emotions.

In [ ]:
X = VIEWS["X_raw"]
# Per-emotion intensity vector: v_int(e) = mean_s (X[s, e, high] - X[s, e, low])
v_int_per_emo = np.stack([X[:, e, HI_IDX, :].mean(0) - X[:, e, LOW_IDX, :].mean(0) for e in range(8)])  # (8, D)
v_int_per_emo_n = v_int_per_emo / (np.linalg.norm(v_int_per_emo, axis=1, keepdims=True) + 1e-12)
shared_cos = v_int_per_emo_n @ v_int_per_emo_n.T
off_diag = shared_cos[~np.eye(8, dtype=bool)]
print("Per-emotion intensity-axis cosine (off-diagonal):")
print(f"  mean   = {off_diag.mean():.3f}")
print(f"  median = {np.median(off_diag):.3f}")
print(f"  min    = {off_diag.min():.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(shared_cos, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            xticklabels=EMOTION_ORDER, yticklabels=EMOTION_ORDER, ax=ax)
ax.set_title(f"Intensity-axis cosine across emotions (L{PRIMARY_LAYER})\nmedian off-diag = {np.median(off_diag):.2f}")
plt.tight_layout(); fig.savefig(OUT_DIR / "phaseD_intensity_axis_cosines.png", bbox_inches="tight")
plt.show()

In [ ]:
# Global intensity axis: average the per-emotion direction vectors.
v_int = v_int_per_emo.mean(axis=0)
v_int_n = v_int / (np.linalg.norm(v_int) + 1e-12)

# Project every (s, e, i) onto v_int_n -> scalar t. Check monotonicity per (s, e).
proj = X.reshape(-1, D) @ v_int_n                                  # (N*8*3,)
proj = proj.reshape(N_SUB, N_EMO, N_INT)
mono = ((proj[:, :, LOW_IDX] < proj[:, :, MED_IDX]) & (proj[:, :, MED_IDX] < proj[:, :, HI_IDX]))
mono_fraction = float(mono.mean())
print(f"Strict-monotonic fraction over (source, emotion) : {mono_fraction:.3f}")

# Distributions per intensity
fig, ax = plt.subplots(figsize=(7, 4))
for label, ii in [("low", LOW_IDX), ("medium", MED_IDX), ("high", HI_IDX)]:
    sns.kdeplot(proj[:, :, ii].ravel(), ax=ax, color=INT_COLORS[label], label=label, fill=True, alpha=0.25)
ax.set_xlabel("⟨X, v_intensity⟩"); ax.set_ylabel("density")
ax.set_title(f"Projection on shared intensity axis (L{PRIMARY_LAYER})\nmonotonic fraction = {mono_fraction:.2f}")
ax.legend(); plt.tight_layout()
fig.savefig(OUT_DIR / "phaseD_projection_kde.png", bbox_inches="tight")
plt.show()

In [ ]:
# Disentanglement check: project out v_int from R_source and re-run an 8-class probe.
R = VIEWS["R_source"]
R_flat = R.reshape(-1, D)
y_emo  = np.tile(np.repeat(np.arange(8), N_INT), N_SUB)
groups = np.repeat(np.arange(N_SUB), N_EMO * N_INT)

R_proj = R_flat - np.outer(R_flat @ v_int_n, v_int_n)            # remove intensity component

def probe_balanced_acc(Xmat, y, grp, n_splits=3, sub=8000):
    rng_l = np.random.default_rng(SEED)
    if Xmat.shape[0] > sub:
        sel = rng_l.choice(Xmat.shape[0], size=sub, replace=False)
        Xmat, y, grp = Xmat[sel], y[sel], grp[sel]
    gkf = GroupKFold(n_splits=n_splits)
    bas = []
    for tr, te in gkf.split(Xmat, y, grp):
        clf = RidgeClassifier(alpha=1.0).fit(Xmat[tr], y[tr])
        bas.append(balanced_accuracy_score(y[te], clf.predict(Xmat[te])))
    return float(np.mean(bas)), float(np.std(bas))

ba_orig, _ = probe_balanced_acc(R_flat, y_emo, groups)
ba_proj, _ = probe_balanced_acc(R_proj, y_emo, groups)
print(f"8-class balanced acc (R_source)                : {ba_orig:.3f}")
print(f"8-class balanced acc (R_source ⟂ v_intensity)  : {ba_proj:.3f}")
print(f"  Δ                                            : {ba_proj - ba_orig:+.3f}")

PHASE_D = {
    "layer": PRIMARY_LAYER,
    "per_emotion_intensity_cos_mean":   float(off_diag.mean()),
    "per_emotion_intensity_cos_median": float(np.median(off_diag)),
    "per_emotion_intensity_cos_min":    float(off_diag.min()),
    "monotonic_fraction": mono_fraction,
    "emotion_ba_R_source":          ba_orig,
    "emotion_ba_R_source_no_vint":  ba_proj,
    "emotion_ba_drop":              float(ba_orig - ba_proj),
}
pd.DataFrame([PHASE_D]).to_csv(OUT_DIR / "phaseD_summary.csv", index=False)
PHASE_D

## Phase E — Clustering, soft membership, interpolation

We look for sub-Plutchik structure (splits), super-Plutchik structure (merges), and we check whether genuinely-blended affect activations exist (soft membership) and what the corresponding texts look like (interpolation lookup).

In [ ]:
# Clustering on R_source flat (subsampled for speed).
R_flat = VIEWS["R_source"].reshape(-1, D)
y_emo_flat = np.tile(np.repeat(np.arange(8), N_INT), N_SUB)
sub = min(8000, R_flat.shape[0])
rng_e = np.random.default_rng(SEED)
sel = rng_e.choice(R_flat.shape[0], size=sub, replace=False)
Xs, ys = R_flat[sel], y_emo_flat[sel]

ks = list(range(2, 17))
results = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xs)
    labels = km.labels_
    sil = silhouette_score(Xs, labels, sample_size=min(3000, sub), random_state=SEED)
    results.append({
        "k": k,
        "silhouette": float(sil),
        "inertia":    float(km.inertia_),
        "ari_vs_plutchik": float(adjusted_rand_score(ys, labels)),
        "nmi_vs_plutchik": float(normalized_mutual_info_score(ys, labels)),
    })
clust_df = pd.DataFrame(results)
clust_df.to_csv(OUT_DIR / "phaseE_cluster_sweep.csv", index=False)
best_k_sil = int(clust_df.loc[clust_df["silhouette"].idxmax(), "k"])
ari_at_8   = float(clust_df.loc[clust_df["k"] == 8, "ari_vs_plutchik"].iloc[0])
print(f"Best k by silhouette : {best_k_sil}")
print(f"ARI at k=8           : {ari_at_8:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(clust_df["k"], clust_df["silhouette"], "o-", color="#4363d8")
axes[0].axvline(8, color="red", ls="--", lw=0.8, label="Plutchik k=8")
axes[0].set_xlabel("k"); axes[0].set_ylabel("silhouette"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(clust_df["k"], clust_df["ari_vs_plutchik"], "o-", color="#3cb44b", label="ARI")
axes[1].plot(clust_df["k"], clust_df["nmi_vs_plutchik"], "s-", color="#f58231", label="NMI")
axes[1].axvline(8, color="red", ls="--", lw=0.8); axes[1].set_xlabel("k"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); fig.savefig(OUT_DIR / "phaseE_cluster_sweep.png", bbox_inches="tight")
plt.show()

# Classify outcome.
if best_k_sil == 8 and ari_at_8 >= 0.7:
    cluster_case = "c_refutes"     # discrete labels match
elif best_k_sil > 8:
    cluster_case = "a_split"       # sub-Plutchik structure
elif best_k_sil < 8:
    cluster_case = "b_merge"       # super-Plutchik (wheel-adjacent) merging
else:
    cluster_case = "c_partial"     # k=8 but ARI low -> geometry differs from labels
print(f"Cluster case verdict : {cluster_case}")

In [ ]:
# Soft membership: cosine similarity to the 8 centroids -> softmax -> entropy.
C = geom["C"]                                      # (8, D)
Cn = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-12)
R_norm = R_flat / (np.linalg.norm(R_flat, axis=1, keepdims=True) + 1e-12)
sim = R_norm @ Cn.T                                # (N_samples, 8)
# softmax with temperature 0.1 (sharp) to get a meaningful entropy distribution
T = 0.1
logits = sim / T
logits -= logits.max(axis=1, keepdims=True)
soft = np.exp(logits); soft /= soft.sum(axis=1, keepdims=True)
entropy = -(soft * np.log(soft + 1e-12)).sum(axis=1)
# top-1 margin (softmax)
top1 = soft.max(axis=1)
soft_sorted = np.sort(soft, axis=1)
margin = soft_sorted[:, -1] - soft_sorted[:, -2]
frac_low_margin = float((margin < 0.05).mean())
mean_entropy = float(entropy.mean())
print(f"Soft-membership mean entropy : {mean_entropy:.3f}  nats (max log 8 = {np.log(8):.3f})")
print(f"Fraction with top1-top2 margin < 0.05 : {frac_low_margin:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(entropy, bins=40, color="#4363d8", alpha=0.85)
ax.axvline(mean_entropy, color="red", ls="--", label=f"mean={mean_entropy:.2f}")
ax.set_xlabel("soft-membership entropy (nats)"); ax.set_ylabel("count")
ax.set_title(f"Soft membership over 8 emotion centroids (L{PRIMARY_LAYER}, T={T})")
ax.legend(); plt.tight_layout()
fig.savefig(OUT_DIR / "phaseE_soft_entropy.png", bbox_inches="tight")
plt.show()

In [ ]:
# Interpolation lookup: build a text registry keyed by (source_id, intensity, emotion).
REGISTRY: dict[tuple[str, str, str], str] = {}
with open(JSONL_PATH) as f:
    for line in f:
        rec = json.loads(line)
        sid = rec["source_id"]
        intens = rec["intensity_level"]
        for emo in EMOTION_ORDER:
            REGISTRY[(sid, intens, emo)] = rec.get(f"{emo}_rewrite", "")
print(f"Registry entries: {len(REGISTRY)}")

# Build flat (text, vector) list aligned with VIEWS["X_raw"] for nearest-neighbour search.
X_flat = VIEWS["X_raw"].reshape(-1, D)
src_idx_flat = np.repeat(np.arange(N_SUB), N_EMO * N_INT)
emo_idx_flat = np.tile(np.repeat(np.arange(N_EMO), N_INT), N_SUB)
int_idx_flat = np.tile(np.arange(N_INT), N_SUB * N_EMO)
# Precompute once (was being recomputed inside nearest_text -> OOM).
X_FLAT_NORM = X_flat / (np.linalg.norm(X_flat, axis=1, keepdims=True) + 1e-12)

def nearest_text(query: np.ndarray, topk: int = 3, restrict_intensity: str | None = None) -> list[tuple[float, str, str, str]]:
    q = query / (np.linalg.norm(query) + 1e-12)
    sim = X_FLAT_NORM @ q
    if restrict_intensity is not None:
        ii = INTENSITY_ORDER.index(restrict_intensity)
        mask = (int_idx_flat == ii)
        sim = np.where(mask, sim, -np.inf)
    top = np.argpartition(-sim, topk)[:topk]
    top = top[np.argsort(-sim[top])]
    out = []
    for idx in top:
        sid = SRC_ID_SUB[src_idx_flat[idx]]
        emo = EMOTION_ORDER[emo_idx_flat[idx]]
        intens = INTENSITY_ORDER[int_idx_flat[idx]]
        out.append((float(sim[idx]), sid, emo, intens, REGISTRY.get((sid, intens, emo), "<MISSING>")))
    return out

# Mean per-emotion vectors at intensity=medium (anchors for interpolation).
anchor = {emo: VIEWS["X_raw"][:, EMO_TO_IDX[emo], MED_IDX, :].mean(axis=0) for emo in EMOTION_ORDER}

def interpolation_test(a: str, b: str, alphas=(0.0, 0.25, 0.5, 0.75, 1.0), topk: int = 2):
    va, vb = anchor[a], anchor[b]
    print(f"\n=== Interpolating  {a}  ↔  {b}  ===")
    for alpha in alphas:
        q = (1 - alpha) * va + alpha * vb
        hits = nearest_text(q, topk=topk, restrict_intensity="medium")
        print(f"\nα={alpha:.2f}  (a={a} → b={b})")
        for sim_v, sid, emo, intens, text in hits:
            t = text[:140].replace("\n", " ")
            print(f"  cos={sim_v:.3f}  src={sid}  label={emo}/{intens}  | {t}")

# Adjacent (joy-trust), distant (joy-fear), opposite (joy-sadness) on Plutchik's wheel.
interpolation_test("joy", "trust")
interpolation_test("joy", "sadness")
interpolation_test("fear", "surprise")

In [ ]:
PHASE_E = {
    "layer": PRIMARY_LAYER,
    "best_k_silhouette": best_k_sil,
    "ari_at_8": ari_at_8,
    "cluster_case": cluster_case,
    "mean_soft_entropy": mean_entropy,
    "frac_low_margin":   frac_low_margin,
}
pd.DataFrame([PHASE_E]).to_csv(OUT_DIR / "phaseE_summary.csv", index=False)
PHASE_E

## Phase F — Auxiliary VAD check

Coarse 2-axis representation of the 8 emotions (Russell circumplex + Plutchik dominance heuristic). If a 2-axis probe reaches a high fraction of the 8-class probe's balanced accuracy, the 8 labels are well-approximated by 2 continuous dimensions.

*This is intentionally a weak, hand-mapped VAD; we do not score the rewrite text with an external model.*

In [ ]:
# Hand mapping (Plutchik → coarse valence/arousal). Documented for reproducibility.
VAD = {
    "joy":          ( 1.0,  0.5),
    "trust":        ( 1.0, -0.5),
    "anticipation": ( 0.5,  0.0),
    "surprise":     ( 0.0,  1.0),
    "fear":         (-0.5,  1.0),
    "anger":        (-1.0,  1.0),
    "sadness":      (-1.0, -1.0),
    "disgust":      (-1.0,  0.0),
}
V = np.array([VAD[e][0] for e in EMOTION_ORDER])    # (8,)
A = np.array([VAD[e][1] for e in EMOTION_ORDER])    # (8,)

# Targets per sample (use R_source emotion identity).
y_v = V[y_emo_flat]    # (N_samples,)
y_a = A[y_emo_flat]

# Use the same subsampled flat matrix for fair comparison with the 8-class probe.
Xs_full = R_flat[sel]; ys_full = y_emo_flat[sel]
yv_s = y_v[sel]; ya_s = y_a[sel]
grp_s = np.repeat(np.arange(N_SUB), N_EMO * N_INT)[sel]

gkf = GroupKFold(n_splits=3)
ba_8, ba_va, ba_va_only_v = [], [], []
for tr, te in gkf.split(Xs_full, ys_full, grp_s):
    clf = RidgeClassifier(alpha=1.0).fit(Xs_full[tr], ys_full[tr])
    ba_8.append(balanced_accuracy_score(ys_full[te], clf.predict(Xs_full[te])))
    rv = Ridge(alpha=1.0).fit(Xs_full[tr], yv_s[tr])
    ra = Ridge(alpha=1.0).fit(Xs_full[tr], ya_s[tr])
    pv = rv.predict(Xs_full[te])
    pa = ra.predict(Xs_full[te])
    # Map back to labels via nearest 2-D anchor.
    preds = []
    for v_, a_ in zip(pv, pa):
        d = (V - v_) ** 2 + (A - a_) ** 2
        preds.append(int(np.argmin(d)))
    ba_va.append(balanced_accuracy_score(ys_full[te], preds))
ba_8_mean   = float(np.mean(ba_8))
ba_va_mean  = float(np.mean(ba_va))
ratio       = ba_va_mean / ba_8_mean if ba_8_mean > 0 else 0.0
print(f"8-class probe BA       : {ba_8_mean:.3f}")
print(f"2-axis VAD probe BA    : {ba_va_mean:.3f}")
print(f"Ratio (2-axis / 8-class) : {ratio:.3f}")

PHASE_F = {
    "layer": PRIMARY_LAYER,
    "ba_8class": ba_8_mean,
    "ba_2axis_vad": ba_va_mean,
    "vad_ratio": ratio,
}
pd.DataFrame([PHASE_F]).to_csv(OUT_DIR / "phaseF_summary.csv", index=False)
PHASE_F

## Phase G — Synthesis, robustness, decision

Headline metrics re-evaluated at layers 10 and 16, then a deterministic verdict.

**Decision rule.** Mark each pillar as supported when:
- **B**: `top-2 EVR ≥ 0.70` AND `wheel_p < 0.05`
- **C**: `mean median pair-cos ≥ 0.5` AND `mean triangle residual ratio ≤ 0.5`
- **D**: `per-emotion intensity cos median ≥ 0.5` AND `monotonic fraction ≥ 0.8`
- **E**: `cluster_case ∈ {a_split, b_merge, c_partial}` OR `mean_soft_entropy ≥ 0.5`

Hypothesis is **supported** when ≥ 3 of 4 pillars hold.

In [ ]:
def headline_metrics_at_layer(layer: int) -> dict:
    V_ = build_views_sub(layer)
    g_ = centroid_geometry(V_["R_source"])
    pdf_, ps_ = pairwise_direction_stats(V_["X_raw"])
    X_ = V_["X_raw"]
    v_pe = np.stack([X_[:, e, HI_IDX, :].mean(0) - X_[:, e, LOW_IDX, :].mean(0) for e in range(8)])
    v_pe_n = v_pe / (np.linalg.norm(v_pe, axis=1, keepdims=True) + 1e-12)
    cosmat = v_pe_n @ v_pe_n.T
    off = cosmat[~np.eye(8, dtype=bool)]
    v_int_ = v_pe.mean(0)
    v_int_n_ = v_int_ / (np.linalg.norm(v_int_) + 1e-12)
    proj_ = X_.reshape(-1, D) @ v_int_n_
    proj_ = proj_.reshape(N_SUB, N_EMO, N_INT)
    mono_ = ((proj_[:, :, LOW_IDX] < proj_[:, :, MED_IDX]) & (proj_[:, :, MED_IDX] < proj_[:, :, HI_IDX])).mean()
    out = {
        "layer": layer,
        "top2_evr": float(g_["evr"][:2].sum()),
        "participation_ratio": float(g_["pr"]),
        "pair_mean_median_cos": float(pdf_["median_cos"].mean()),
        "per_emo_int_cos_median": float(np.median(off)),
        "monotonic_fraction": float(mono_),
    }
    del V_; gc.collect()
    return out

robust_rows = [headline_metrics_at_layer(PRIMARY_LAYER)]
for L in ROBUST_LAYERS:
    print(f"... layer {L}")
    robust_rows.append(headline_metrics_at_layer(L))
robust_df = pd.DataFrame(robust_rows)
robust_df.to_csv(OUT_DIR / "phaseG_robustness.csv", index=False)
robust_df

In [ ]:
# UMAP illustration (one figure, two panels).
if HAS_UMAP:
    sub_u = min(5000, R_flat.shape[0])
    rng_u = np.random.default_rng(SEED)
    sel_u = rng_u.choice(R_flat.shape[0], size=sub_u, replace=False)
    reducer = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=30, min_dist=0.1, metric="cosine")
    emb = reducer.fit_transform(R_flat[sel_u])
    ye = y_emo_flat[sel_u]
    yi = np.tile(np.arange(N_INT), N_SUB * N_EMO)[sel_u]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ei, emo in enumerate(EMOTION_ORDER):
        m = ye == ei
        axes[0].scatter(emb[m, 0], emb[m, 1], s=4, alpha=0.5, color=EMO_COLORS[ei], label=emo, linewidths=0)
    axes[0].set_title("UMAP(R_source) — colored by emotion"); axes[0].legend(fontsize=7, ncol=2)
    for ii, intens in enumerate(INTENSITY_ORDER):
        m = yi == ii
        axes[1].scatter(emb[m, 0], emb[m, 1], s=4, alpha=0.5, color=INT_COLORS[intens], label=intens, linewidths=0)
    axes[1].set_title("UMAP(R_source) — colored by intensity"); axes[1].legend()
    plt.tight_layout(); fig.savefig(OUT_DIR / "phaseG_umap.png", bbox_inches="tight")
    plt.show()
else:
    print("umap-learn not available — skipping UMAP figure.")

In [ ]:
# Verdict.
pillar_B = (PHASE_B["top2_evr"] >= 0.70) and (PHASE_B["wheel_p_value"] < 0.05)
pillar_C = (pair_summary["mean_median_cos_pairs"] >= 0.5) and (tri_summary["mean_res_ratio"] <= 0.5)
pillar_D = (PHASE_D["per_emotion_intensity_cos_median"] >= 0.5) and (PHASE_D["monotonic_fraction"] >= 0.8)
pillar_E = (cluster_case in {"a_split", "b_merge", "c_partial"}) or (PHASE_E["mean_soft_entropy"] >= 0.5)
n_pillars = sum([pillar_B, pillar_C, pillar_D, pillar_E])
supported = n_pillars >= 3

summary = {
    "layer": PRIMARY_LAYER,
    "phase_B": PHASE_B,
    "phase_C": {**pair_summary, **tri_summary},
    "phase_D": PHASE_D,
    "phase_E": PHASE_E,
    "phase_F": PHASE_F,
    "robustness": robust_df.to_dict(orient="records"),
    "pillars": {"B": pillar_B, "C": pillar_C, "D": pillar_D, "E": pillar_E},
    "n_pillars_passed": int(n_pillars),
    "hypothesis_supported": bool(supported),
}
with open(OUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=float)

print("=" * 60)
print(f"  Continuous-affect hypothesis  (layer {PRIMARY_LAYER})")
print("=" * 60)
for name, ok in summary["pillars"].items():
    print(f"  pillar {name}: {'PASS' if ok else 'FAIL'}")
print("-" * 60)
print(f"  Pillars passed       : {n_pillars} / 4")
print(f"  Hypothesis supported : {supported}")
print("=" * 60)
print(f"\nArtifacts in: {OUT_DIR}")